# Field Shift · Phase 2 checks for the API expansion

Rules written and committed before this notebook ran: [`docs/ML_API_VALIDATION.md`](../../docs/ML_API_VALIDATION.md).
Tested models only (read-only). Writes `artifacts/api_validation/`. Nothing is tuned after seeing results.

## 1. Setup and tables (same `fs_ml` code as every phase)

In [1]:
import sys, json, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb
from scipy.stats import spearmanr
warnings.filterwarnings("ignore", message="(All-NaN slice|Mean of empty slice)")
ML_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ML_DIR))
from fs_ml import data as D, features as F, evaluate as E, project as P
from fs_ml.predict import load

SEED, N_BOOT = 42, 1000
ART = ML_DIR / "artifacts" / "api_validation"; ART.mkdir(parents=True, exist_ok=True)
SM, ND = load("sm_root_monthly"), load("ndvi_quarterly")
RMSE_SM = SM["card"]["metrics"]["validation_2018_2022"]["RMSE"]["value"]
CARD = json.loads((ML_DIR / "artifacts" / "projections" / "projection_card.json").read_text())
CORR = {k: {int(a): b for a, b in v.items()} for k, v in CARD["correction_by_lead"].items()}
POOL, UNTIL = list(range(1998, 2018)), "2017-12-01"

def build(load_years, suffix, keep):
    monthly, _ = D.load_gee_monthly(load_years); veg, _ = D.load_gee_quarterly(load_years)
    pw = {p: D.load_power(f"power_{p}{suffix}", load_years)[0] for p in ["t2m", "rh2m", "ws2m", "allsky"]}
    merra = pw["t2m"].merge(pw["rh2m"], on=["power_lat", "power_lon", "date"]).merge(pw["ws2m"], on=["power_lat", "power_lon", "date"])
    gaez = {k: D.load_gaez(k)[0] for k in F.GAEZ_FEATURES}
    cells = F.static_cells(D.cells_from_ids(monthly.cell_id), gaez, merra[["power_lat", "power_lon"]].drop_duplicates().reset_index(drop=True),
                           pw["allsky"][["power_lat", "power_lon"]].drop_duplicates().reset_index(drop=True))
    mp = F.build_monthly_panel(monthly, cells, merra, pw["allsky"]); qp = F.build_quarterly_panel(mp, veg)
    extra = pd.DataFrame({"cell_id": cells.cell_id.to_numpy(), "block": np.asarray(E.spatial_blocks(cells.lat, cells.lon)),
                          "province": np.asarray(D.provinces_of(cells))})
    return mp.merge(extra, on="cell_id")[lambda d: d.year.between(*keep)], qp.merge(extra, on="cell_id")[lambda d: d.year.between(*keep)]

parts = [build((1998, 2017), "", (1998, 2017)), build((2017, 2022), "_2018_2022", (2018, 2022)), build((2022, 2025), "_2023_2025", (2023, 2025))]
MP = pd.concat([p[0] for p in parts], ignore_index=True); QP = pd.concat([p[1] for p in parts], ignore_index=True)
CUBE = P.build_cube(MP, QP)
VALID = ~np.isnan(CUBE.v["sm_root_m3m3"]).all(axis=1)
INSIDE = (CUBE.cells.province != "outside").to_numpy() & VALID
print(len(CUBE.cells), "cells |", int(INSIDE.sum()), "inside Lambayeque with soil water | RMSE tolerance", round(RMSE_SM, 4))

3149 cells | 1950 inside Lambayeque with soil water | RMSE tolerance 0.0125


## 2. Scenario runs from the 49 starts (Dec 2017 – Dec 2021)

In [2]:
def cut(w, h):
    return {k: (v[:, :, :h] if isinstance(v, np.ndarray) and v.ndim == 3 else v) for k, v in w.items()}
def med(w, s, h):
    return np.median(P.project_sm(SM, CUBE, s, h, None, weather=cut(w, h))["members"][:, INSIDE, :], axis=0)

RUNS = []
t0 = time.time()
for s in pd.date_range("2017-12-01", "2021-12-01", freq="MS"):
    w = P.analog_weather(CUBE, s, 12, POOL, until=UNTIL)
    if len(w["years"]) < 5: continue
    base = med(w, s, 12)
    r = {"start": s, "base": base,
         "t_plus": med(P.scenario_adjust(w, t2m_delta=2.0), s, 3), "t_minus": med(P.scenario_adjust(w, t2m_delta=-2.0), s, 3),
         "rain_half": med(P.scenario_adjust(w, rain_factor=0.5), s, 3), "rain_double": med(P.scenario_adjust(w, rain_factor=2.0), s, 3)}
    for n in range(1, 7):
        r[f"dry{n}"] = med(P.scenario_drought(w, list(range(1, n + 1)), CUBE, s), s, 12)
    RUNS.append(r)
print(len(RUNS), "starts |", round(time.time() - t0), "s")

49 starts | 492 s


## 3. S1–S4: scenario behaviour

In [3]:
TOL, STRICT = RMSE_SM, 1e-4
stack = lambda key, lead: np.concatenate([r[key][:, lead - 1] - r["base"][:, lead - 1] for r in RUNS])
CHECKS = {}

d_plus, d_minus = stack("t_plus", 3), stack("t_minus", 3)
CHECKS["S1"] = {"feature": "temperature what-if", "raise_share_plus2": float((d_plus > TOL).mean()), "fall_share_minus2": float((d_minus < -TOL).mean()),
                "strict_raise_share_plus2": float((d_plus > STRICT).mean()), "mean_effect_plus2": float(d_plus.mean()), "mean_effect_minus2": float(d_minus.mean())}
CHECKS["S1"]["pass"] = CHECKS["S1"]["raise_share_plus2"] <= 0.01 and CHECKS["S1"]["fall_share_minus2"] <= 0.01

rows = []
for n in range(1, 7):
    end = stack(f"dry{n}", n); l6 = stack(f"dry{n}", 6)
    rows.append({"months": n, "mean_change_end": float(end.mean()), "mean_change_lead6": float(l6.mean()),
                 "rise_share_end": float((end > TOL).mean()), "strict_rise_share_end": float((end > STRICT).mean())})
S2 = pd.DataFrame(rows)
mono = all(S2.mean_change_lead6.iloc[i + 1] <= S2.mean_change_lead6.iloc[i] + 0.001 for i in range(5))
CHECKS["S2"] = {"feature": "droughts of 1-6 months", "table": rows, "monotone_lead6": mono,
                "pass": bool((S2.mean_change_end < 0).all() and mono and (S2.rise_share_end <= 0.01).all())}

dh, dd = stack("rain_half", 3), stack("rain_double", 3)
CHECKS["S3"] = {"feature": "rain-only what-if", "half_raise_share": float((dh > TOL).mean()), "double_lower_share": float((dd < -TOL).mean()),
                "mean_effect_half": float(dh.mean()), "mean_effect_double": float(dd.mean())}
CHECKS["S3"]["pass"] = (CHECKS["S3"]["half_raise_share"] <= 0.01 and CHECKS["S3"]["double_lower_share"] <= 0.01
                        and CHECKS["S3"]["mean_effect_half"] <= -0.003 and CHECKS["S3"]["mean_effect_double"] >= 0.003)

d3, d12 = stack("dry3", 3), stack("dry3", 12); ok = np.abs(d3) > 1e-6
CHECKS["S4"] = {"feature": "recovery after a 3-month drought", "share_forgetting": float((np.abs(d12[ok]) < np.abs(d3[ok])).mean()),
                "mean_change_lead3": float(d3.mean()), "mean_change_lead12": float(d12.mean())}
CHECKS["S4"]["pass"] = CHECKS["S4"]["share_forgetting"] >= 0.90
pd.DataFrame(rows).round(4)

,months,mean_change_end,mean_change_lead6,rise_share_end,strict_rise_share_end
0,1,-0.0024,-0.0014,0.0000,0.3421
1,2,-0.0117,-0.0029,0.0000,0.3236
2,3,-0.0168,-0.0055,0.0012,0.3292
3,4,-0.0199,-0.0105,0.0055,0.3175
4,5,-0.0216,-0.0217,0.0138,0.3081
5,6,-0.0234,-0.0234,0.0248,0.2954


In [4]:
{k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items() if kk != "table"} for k, v in CHECKS.items()}

{'S1': {'feature': 'temperature what-if',
  'raise_share_plus2': 0.0973,
  'fall_share_minus2': 0.0057,
  'strict_raise_share_plus2': 0.5476,
  'mean_effect_plus2': 0.0024,
  'mean_effect_minus2': 0.0013,
  'pass': False},
 'S2': {'feature': 'droughts of 1-6 months',
  'monotone_lead6': True,
  'pass': False},
 'S3': {'feature': 'rain-only what-if',
  'half_raise_share': 0.0015,
  'double_lower_share': 0.0046,
  'mean_effect_half': -0.0006,
  'mean_effect_double': 0.0006,
  'pass': False},
 'S4': {'feature': 'recovery after a 3-month drought',
  'share_forgetting': 0.8309,
  'mean_change_lead3': -0.0168,
  'mean_change_lead12': -0.0015,
  'pass': False}}

## 4. S5 drought risk and S6 alerts: hindcasts on 2018–2022

In [5]:
YEARS = CUBE.dates.year.to_numpy()
P20 = {m: np.nanpercentile(CUBE.v["sm_root_m3m3"][:, (CUBE.dates.month == m) & (YEARS <= 2017)], 20, axis=1) for m in range(1, 13)}
risk_p, risk_y, alert_sm = [], [], []
for s in pd.date_range("2017-12-01", "2022-11-01", freq="MS"):
    h = min(3, (2022 - s.year) * 12 + 12 - s.month)
    run = P.project_sm(SM, CUBE, s, h, POOL, until=UNTIL); mem = run["members"][:, INSIDE, :]
    for k, d in enumerate(run["dates"]):
        y = CUBE.v["sm_root_m3m3"][INSIDE, CUBE.t(d)]; p20 = P20[d.month][INSIDE]; ok = ~np.isnan(y)
        risk_p.append((mem[:, ok, k] < p20[ok]).mean(axis=0)); risk_y.append((y[ok] < p20[ok]).astype(float))
        if k == 0:
            lo = np.quantile(mem[:, ok, 0], 0.05, axis=0) - CORR["sm"][1]
            alert_sm.append(y[ok] < lo)
rp, ry = np.concatenate(risk_p), np.concatenate(risk_y)
bins = np.minimum((rp * 5).astype(int), 4); calib = []
for b in range(5):
    m = bins == b
    if m.sum(): calib.append({"bin": f"{b/5:.1f}-{(b+1)/5:.1f}", "cases": int(m.sum()), "mean_predicted": float(rp[m].mean()), "observed": float(ry[m].mean())})
bs, bs_ref = float(((rp - ry) ** 2).mean()), float(((0.2 - ry) ** 2).mean())
CAL = pd.DataFrame(calib)
CHECKS["S5"] = {"feature": "drought risk (share of analogs below the cell's 20th percentile)", "calibration": calib, "brier": bs, "brier_ref_0_2": bs_ref,
                "brier_skill": 1 - bs / bs_ref, "max_gap": float((CAL[CAL.cases >= 1000].observed - CAL[CAL.cases >= 1000].mean_predicted).abs().max())}
CHECKS["S5"]["pass"] = CHECKS["S5"]["max_gap"] <= 0.10 and CHECKS["S5"]["brier_skill"] > 0
fs = float(np.concatenate(alert_sm).mean())

alert_nd = []
for s in [p.to_timestamp(how="end").normalize().replace(day=1) for p in pd.period_range("2017Q4", "2022Q3", freq="Q")]:
    sm = P.project_sm(SM, CUBE, s, 3, POOL, until=UNTIL); nd = P.project_ndvi(ND, CUBE, sm)
    q = nd["quarters"][0]; y = CUBE.ndvi[INSIDE, CUBE.quarters.get_loc(q)]; ok = ~np.isnan(y)
    lo = np.quantile(nd["members"][:, INSIDE, 0][:, ok], 0.05, axis=0) - CORR["ndvi"][1]
    alert_nd.append(y[ok] < lo)
fn = float(np.concatenate(alert_nd).mean())
CHECKS["S6"] = {"feature": "alerts below the lead-1 band", "flag_share_soil_water": fs, "flag_share_ndvi": fn,
                "pass": 0.025 <= fs <= 0.10 and 0.025 <= fn <= 0.10}
display(CAL.round(3)); print({k: round(v, 4) for k, v in CHECKS["S5"].items() if isinstance(v, float)}, "| alerts:", round(fs, 4), round(fn, 4))

,bin,cases,mean_predicted,observed
0,0.0-0.2,199561,0.025,0.058
1,0.2-0.4,46238,0.266,0.304
2,0.4-0.6,24097,0.472,0.318
3,0.6-0.8,18508,0.681,0.346
4,0.8-1.0,56746,0.951,0.655


{'brier': 0.1431, 'brier_ref_0_2': 0.1736, 'brier_skill': 0.1761, 'max_gap': 0.3356} | alerts: 0.0472 0.0369


## 5. S7 NDVI greener than expected (descriptive only), S8 cells outside Lambayeque, S10 per-cell confidence

In [6]:
def one_step(df, m):
    t = m["card"]["target"]["column"]; d = df[df[t].notna()].copy(); d["pred"] = m["model"].predict(d[m["features"]]); d["y"] = d[t]
    return d
QV = one_step(QP[QP.year >= 2018], ND)
RMSE_ND = ND["card"]["metrics"]["validation_2018_2022"]["RMSE"]["value"]
QV["res"] = QV.y - QV.pred
recent = QV[QV.year >= 2024].groupby("cell_id").res.apply(lambda r: int((r > RMSE_ND).sum()))
persist = recent[recent >= 3]
prov = QV.drop_duplicates("cell_id").set_index("cell_id").province
CHECKS["S7"] = {"feature": "NDVI greener than expected", "pass": False, "status": "not testable: no independent irrigated-area data",
                "cells_with_3plus_of_8_quarters_above_error_2024_2025": int(len(persist)), "share_of_cells": float(len(persist) / recent.size),
                "by_province": prov.loc[persist.index].value_counts().to_dict()}

def criteria(df, m, name, period):
    t = m["card"]["target"]["column"]; d = one_step(df[df.year.between(*period)], m).reset_index(drop=True)
    tr = MP if name == "sm" else QP; keys, season = (["cell_id", "month"], "month") if name == "sm" else (["cell_id", "quarter"], "quarter")
    clim = E.predict_climatology(E.climatology(tr[tr.year <= 2017], t, keys, season), d)
    lag = "sm_root_m3m3_lag1" if name == "sm" else "ndvi_lag1"; pers = np.where(d[lag].isna(), clim, d[lag])
    ci = E.block_bootstrap(d.y.to_numpy(), {"model": d.pred.to_numpy(), "climatology": clim, "persistence": pers}, d.block.to_numpy(), "model", N_BOOT, SEED).set_index(["model", "metric"])
    best = min(["climatology", "persistence"], key=lambda k: ci.loc[(k, "RMSE"), "value"])
    sk = ci.loc[("model", f"skill vs {best}")]
    base = np.asarray(clim if best == "climatology" else pers, float); yr = d.year.to_numpy()
    worst = min(E.skill(E.regression_metrics(d.y[yr == y_], d.pred[yr == y_])["RMSE"], E.regression_metrics(d.y[yr == y_], base[yr == y_])["RMSE"]) for y_ in np.unique(yr))
    thr = {"sm": (0.20, 0.04, 0.01), "ndvi": (0.10, None, 0.03)}[name]
    bias, rmse = ci.loc[("model", "bias"), "value"], ci.loc[("model", "RMSE"), "value"]
    res = {"rows": len(d), "best_rule": best, "skill": float(sk.value), "skill_ci": [float(sk.ci_low), float(sk.ci_high)], "rmse": float(rmse), "bias": float(bias), "worst_year_skill": float(worst)}
    res["pass"] = bool(sk.value >= thr[0] and sk.ci_low > 0 and (thr[1] is None or rmse <= thr[1]) and abs(bias) <= thr[2] and worst >= 0)
    return res
out = {"sm_validation": criteria(MP[MP.province == "outside"], SM, "sm", (2018, 2022)), "ndvi_validation": criteria(QP[QP.province == "outside"], ND, "ndvi", (2018, 2022)),
       "sm_testing_report": criteria(MP[MP.province == "outside"], SM, "sm", (2023, 2025)), "ndvi_testing_report": criteria(QP[QP.province == "outside"], ND, "ndvi", (2023, 2025))}
CHECKS["S8"] = {"feature": "cells outside Lambayeque", **out, "pass": out["sm_validation"]["pass"] and out["ndvi_validation"]["pass"]}

MV = one_step(MP[MP.year >= 2018], SM)
MV["pix"] = (MV.lat / 0.1).round().astype(int).astype(str) + "," + (MV.lon / 0.1).round().astype(int).astype(str)
g = MV.assign(per=np.where(MV.year <= 2022, "val", "test"), se=(MV.pred - MV.y) ** 2).groupby(["pix", "per"]).se.mean().unstack() ** 0.5
rho = spearmanr(g.val, g.test, nan_policy="omit").statistic
CHECKS["S10"] = {"feature": "per-cell confidence", "pixel_groups": int(g.dropna().shape[0]), "spearman_val_vs_test_rmse": float(rho), "pass": bool(rho >= 0.5)}
CHECKS["S9"] = {"feature": "adaptive ranges (CQR + monthly recalibration)", "pass": False, "status": "deferred: needs complete 2026 data"}
{k: v.get("pass") for k, v in sorted(CHECKS.items())}

{'S1': False,
 'S10': True,
 'S2': False,
 'S3': False,
 'S4': False,
 'S5': False,
 'S6': True,
 'S7': False,
 'S8': True,
 'S9': False}

In [7]:
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk not in ("table", "calibration")} for k, v in CHECKS.items()}, indent=1, default=str)[:6000])

{
 "S1": {
  "feature": "temperature what-if",
  "raise_share_plus2": 0.09729984301412872,
  "fall_share_minus2": 0.005693354264782837,
  "strict_raise_share_plus2": 0.5475771847200419,
  "mean_effect_plus2": 0.002416709259362947,
  "mean_effect_minus2": 0.0012723015238833577,
  "pass": false
 },
 "S2": {
  "feature": "droughts of 1-6 months",
  "monotone_lead6": true,
  "pass": false
 },
 "S3": {
  "feature": "rain-only what-if",
  "half_raise_share": 0.0014756671899529043,
  "double_lower_share": 0.0046363160648874935,
  "mean_effect_half": -0.0006221241054423875,
  "mean_effect_double": 0.0005867312083236214,
  "pass": false
 },
 "S4": {
  "feature": "recovery after a 3-month drought",
  "share_forgetting": 0.830922071940036,
  "mean_change_lead3": -0.01680335652829776,
  "mean_change_lead12": -0.0014931108434511565,
  "pass": false
 },
 "S5": {
  "feature": "drought risk (share of analogs below the cell's 20th percentile)",
  "brier": 0.14305071326730562,
  "brier_ref_0_2": 0.17361

## 6. Save

In [8]:
(ART / "checks.json").write_text(json.dumps({"rules": "docs/ML_API_VALIDATION.md", "model_sha256": {"sm_root_monthly": SM["card"]["sha256"], "ndvi_quarterly": ND["card"]["sha256"]},
                                             "checks": CHECKS}, indent=1, default=str))
S2.to_csv(ART / "s2_droughts.csv", index=False); CAL.to_csv(ART / "s5_calibration.csv", index=False)
print(sorted(p.name for p in ART.iterdir()))

['checks.json', 's2_droughts.csv', 's5_calibration.csv']
